In [1]:
from pathlib import Path
import sys
import torch
import json

repo_root = Path.cwd()
if not (repo_root / "neps_nos").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

In [8]:
import neps

folder = repo_root / "neps_runs/46_LI_space"
results_folder = folder / "results"

best_results = (float('inf'), "")
for file in results_folder.glob("*.json"):
    print(f"Checking file: {file.name}", flush=True, end="\r")
    if "Anton" not in file.name and "am" not in file.name:# and "add" in file.name:
        with open(file, "r") as f:
            data = json.load(f)
        if data["incumbent_history"][-1] < best_results[0]:
            best_results = (data["incumbent_history"][-1], file.name)
print(f"Best result: {best_results[0]} from file: {best_results[1]}")

config_path = results_folder.parent / "neps" / best_results[1][:-7] / f"seed_{best_results[1][-6:-5]}"

csv_path = config_path / "summary" / "short.csv"
csv_file = csv_path.open("r")
csv_content = csv_file.read()
print(csv_content.splitlines()[3][len("best_config_id,"):])
id = csv_content.splitlines()[3][len("best_config_id,"):]

pipeline = neps.load_config(config_path= config_path / "configs" / f"{id}" / "config.yaml")
print(pipeline)
x=torch.zeros(1, requires_grad=True)
y=2*x
y.backward()
opt= pipeline["optimizer_cls"]([x])
print(opt)
opt.step()
print(x)

Best result: 3.561625671386719 from file: sota_li1_rand_01_ae_mul_3.json
441_rung_1
{'fidelity': 50, 'learning_rate': 2.1971211698905724e-05, 'optimizer_cls': <class 'neps_nos.neps_nos_space.AdamWExtend.create_optimizer.<locals>.CustomOptimizer'>}
CustomOptimizer(
  Parameter group:
    lr: 0.001
    vars: (0.1, 0.1)
    weight_decay: 0.01
)
Lines:
(fixed) m = beta1 * m + (1 - beta1) * g
(fixed) v = beta2 * v + (1 - beta2) * (g * g)
(fixed) m_hat = m / (1 - beta1)
(fixed) v_hat = v / (1 - beta2)
  v1 = lerp({weight=0.01}, v1, w)
  v1 = add(v2, v1)
   u = square(v1)
(fixed) w = w - lr * (m_hat / (sqrt(v_hat) + eps)) * u

tensor([-3.9601e-05], requires_grad=True)


In [ ]:
import neps_nos.neps_nos_space as neps_nos_space
from neps_nos.neps_config import get_space_basename_and_kwargs, get_space_base_callable

space_base_name, space_kwargs = get_space_basename_and_kwargs("AdamMore")
space_class = get_space_base_callable(space_base_name)
if "fidelity" in space_kwargs and space_kwargs["fidelity"] is True:
    space_kwargs["fidelity"] = (10, 50)
pipeline_space = space_class(**space_kwargs)


In [2]:

# print(hasattr(pipeline_space, "weight_decay"))
x=torch.zeros(1, requires_grad=True)
y=2*x
y.backward()
sample = pipeline_space.sample()
opt= sample[1]["optimizer_cls"]([x])
# print(opt.get_lines())
opt.step()
print(x)


{'t': tensor(0.), 'depth': tensor(0.), 'layer_type_attention': tensor(0.)}
Evaluating line: ('v1', (functools.partial(<built-in method mul of type object at 0x14819c179b40>, other=0.9), 'g'))
Evaluating line: ('v2', (functools.partial(<built-in method clamp of type object at 0x14819c179b40>, min=0, max=0), 'g'))
Evaluating line: ('u1', (<built-in method exp of type object at 0x14819c179b40>, 'g'))
tensor([-0.0084], requires_grad=True)
